# Convert Parquet to CSV

Read one month of NYC Yellow Taxi data from `data/raw` and save it as a CSV file in `data/processed`.

In [171]:
import pandas as pd
import seaborn as sns

from config import PROCESSED_DIR, RAW_DIR, build_dataset_filename

In [172]:
month = "2025-01"

parquet_path = RAW_DIR / build_dataset_filename(month)
csv_path = PROCESSED_DIR / parquet_path.with_suffix(".csv").name

print(parquet_path)
print(csv_path)

/home/localadmin/BOOT_CAMP_exercise/module_3/ML_model_deployment/data/raw/yellow_tripdata_2025-01.parquet
/home/localadmin/BOOT_CAMP_exercise/module_3/ML_model_deployment/data/processed/yellow_tripdata_2025-01.csv


## Read the Parquet file

In [326]:
df = pd.read_parquet(parquet_path)

print(df.shape)
df.head()

(3475226, 20)


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,1,2025-01-01 00:18:38,2025-01-01 00:26:59,1.0,1.60,1.0,N,229,237,1,10.0,3.5,0.5,3.00,0.0,1.0,18.00,2.5,0.0,0.0
1,1,2025-01-01 00:32:40,2025-01-01 00:35:13,1.0,0.50,1.0,N,236,237,1,5.1,3.5,0.5,2.02,0.0,1.0,12.12,2.5,0.0,0.0
2,1,2025-01-01 00:44:04,2025-01-01 00:46:01,1.0,0.60,1.0,N,141,141,1,5.1,3.5,0.5,2.00,0.0,1.0,12.10,2.5,0.0,0.0
3,2,2025-01-01 00:14:27,2025-01-01 00:20:01,3.0,0.52,1.0,N,244,244,2,7.2,1.0,0.5,0.00,0.0,1.0,9.70,0.0,0.0,0.0
4,2,2025-01-01 00:21:34,2025-01-01 00:25:06,3.0,0.66,1.0,N,244,116,2,5.8,1.0,0.5,0.00,0.0,1.0,8.30,0.0,0.0,0.0


In [341]:
import numpy as np

df1 = df.copy()


df1 = df1[~df1["payment_type"].isin([3, 4, 5, 6])].reset_index(drop=True)


df2 = df1.drop(columns=["passenger_count", "RatecodeID", "store_and_fwd_flag","payment_type"])


df2["duration_min"] = ((df2["tpep_dropoff_datetime"] - df2["tpep_pickup_datetime"]).dt.total_seconds() / 60).round(2)

df2 = df2[df2["duration_min"].between(1, 180)].reset_index(drop=True)   # keep trips from 1 minute to 3 hours

df2 = df2.drop(columns=["tpep_pickup_datetime", "tpep_dropoff_datetime"])

df2 = df2[(df2["trip_distance"] >= 0) & (df2["trip_distance"] <=150)].reset_index(drop=True)


df2 = df2[(df2["fare_amount"] >= 0) &  (df2["fare_amount"] <= 600)].reset_index(drop=True)

df2 = df2[(df2["Airport_fee"] >= 0)].reset_index(drop=True)


df2 = df2[df2["total_amount"] >= 0].reset_index(drop=True)


parts = [
    "fare_amount", "extra", "mta_tax", "tip_amount", "tolls_amount",
    "improvement_surcharge", "congestion_surcharge", "Airport_fee", "cbd_congestion_fee",
]

df2["total_amount"] = df2[parts].sum(axis=1).round(2)

df2 = df2.drop(columns=["total_amount"])




In [342]:
## Save as csv file

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
df2.to_csv(csv_path, index=False)

print(f"Saved {len(df2):,} rows to {csv_path}")

Saved 2,795,902 rows to /home/localadmin/BOOT_CAMP_exercise/module_3/ML_model_deployment/data/processed/yellow_tripdata_2025-01.csv
